In [ ]:
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import seaborn as sns

In [ ]:
df = pd.read_csv('train.csv')
df

In [ ]:
df.drop(columns=['id'], inplace=True)
df.head()

In [ ]:
df.isna().sum()

In [ ]:
df.duplicated().sum()

In [ ]:
sns.countplot(x='diagnosed_diabetes', data=df)

In [ ]:
df.columns

In [ ]:
categorical_features = []
numeric_features = []
for column in df.columns:
    if df[column].dtype == 'object':
        categorical_features.append(column)
    else:
        numeric_features.append(column)

In [ ]:
numeric_features

In [ ]:
categorical_features

In [ ]:
sns.countplot(x='gender', data=df, hue='diagnosed_diabetes')

In [ ]:
df['ethnicity'].value_counts()

In [ ]:
sns.countplot(x='ethnicity', data=df, hue='diagnosed_diabetes')

In [ ]:
df['education_level'].value_counts()

In [ ]:
sns.countplot(x='education_level', data=df, hue='diagnosed_diabetes')

In [ ]:
df['income_level'].value_counts()

In [ ]:
sns.countplot(x='income_level', data=df, hue='diagnosed_diabetes')

In [ ]:
df['smoking_status'].value_counts()

In [ ]:
sns.countplot(x='smoking_status', data=df, hue='diagnosed_diabetes')

In [ ]:

from scipy.stats import chi2_contingency
from sklearn.feature_selection import mutual_info_classif
from sklearn.preprocessing import LabelEncoder
categorical_cols = categorical_features  # use all categorical features

target_col = 'diagnosed_diabetes'   # change if needed

results = []

for col in categorical_cols:
    temp = df[[col, target_col]].dropna()

    # --- Chi-square test ---
    contingency = pd.crosstab(temp[col], temp[target_col])
    chi2, p_value, _, _ = chi2_contingency(contingency)

    # --- Cram?r?s V ---
    n = contingency.sum().sum()
    phi2 = chi2 / n
    r, k = contingency.shape
    phi2corr = max(0, phi2 - ((k - 1) * (r - 1)) / (n - 1))
    rcorr = r - ((r - 1) ** 2) / (n - 1)
    kcorr = k - ((k - 1) ** 2) / (n - 1)
    cramers_v = np.sqrt(phi2corr / min((kcorr - 1), (rcorr - 1)))

    # --- Mutual Information ---
    le = LabelEncoder()
    X_encoded = le.fit_transform(temp[col])

    mi = mutual_info_classif(
        X_encoded.reshape(-1, 1),
        temp[target_col],
        discrete_features=True
    )[0]

    # --- Decision logic ---
    if p_value < 0.05 and cramers_v >= 0.10:
        decision = "KEEP"
    elif p_value < 0.05 or mi > 0.01:
        decision = "TEST FURTHER"
    else:
        decision = "DROP"

    results.append([
        col,
        round(p_value, 5),
        round(cramers_v, 3),
        round(mi, 3),
        decision
    ])

# --- Final result table ---
result_df = pd.DataFrame(
    results,
    columns=[
        "feature",
        "chi_square_p_value",
        "cramers_v",
        "mutual_information",
        "decision"
    ]
)

result_df


In [ ]:
numeric_features

In [ ]:
df.describe()


In [ ]:
df[numeric_features].corr()[target_col].sort_values(ascending=False)


In [ ]:
sns.heatmap(df[numeric_features].corr()[target_col].to_frame(), annot=True, cmap='coolwarm')

In [ ]:
from sklearn.preprocessing import LabelEncoder
for col in categorical_features:
    le = LabelEncoder()
    df[col] = le.fit_transform(df[col].astype(str))
    

In [ ]:
df.info()

In [ ]:
x_train = df.drop(columns=[target_col])
y_train = df[target_col]

In [ ]:
# from sklearn.model_selection import train_test_split
# x_train, x_test, y_train, y_test = train_test_split(x, y, test_size=0.2, random_state=42, stratify=y)

In [ ]:
numeric_features = [
    col for col in numeric_features 
    if col != 'diagnosed_diabetes'
]


In [ ]:
from sklearn.preprocessing import StandardScaler

scaler = StandardScaler()

x_train[numeric_features] = scaler.fit_transform(x_train[numeric_features])
# x_test[numeric_features] = scaler.transform(x_test[numeric_features])

In [ ]:
from sklearn.linear_model import LogisticRegression

log_base = LogisticRegression(max_iter=1000)
log_base.fit(x_train, y_train)


In [ ]:
# y_pred = log_base.predict(x_test)
# y_pred

In [ ]:
# from sklearn.metrics import accuracy_score, roc_auc_score, classification_report

# print("Accuracy:", accuracy_score(y_test, y_pred))
# print("AUC:", roc_auc_score(y_test, log_base.predict_proba(x_test)[:,1]))
# print(classification_report(y_test, y_pred))


In [ ]:
# from sklearn.ensemble import RandomForestClassifier

# rf = RandomForestClassifier(
#     n_estimators=300,
#     random_state=42
# )

# rf.fit(x_train, y_train)


In [ ]:
# y_pred = rf.predict(x_test)


In [ ]:
# print("Accuracy:", accuracy_score(y_test, y_pred))
# print("AUC:", roc_auc_score(y_test, rf.predict_proba(x_test)[:,1]))
# print(classification_report(y_test, y_pred))

In [ ]:
# from xgboost import XGBClassifier

# xgb = XGBClassifier(
#     n_estimators=300,
#     max_depth=4,
#     learning_rate=0.05,
#     subsample=0.8,
#     colsample_bytree=0.8,
#     eval_metric='auc',
#     random_state=42
# )

# xgb.fit(x_train, y_train)


In [ ]:
# y_pred = xgb.predict(x_test)

In [ ]:
# print("Accuracy:", accuracy_score(y_test, y_pred))
# print("AUC:", roc_auc_score(y_test, xgb.predict_proba(x_test)[:,1]))
# print(classification_report(y_test, y_pred))

In [ ]:
from catboost import CatBoostClassifier

cat = CatBoostClassifier(
    iterations=500,
    depth=6,
    learning_rate=0.05,
    loss_function='Logloss',
    eval_metric='AUC',
    verbose=0
)

cat.fit(x_train, y_train)


In [ ]:
# y_pred = cat.predict(x_test)
# print("Accuracy:", accuracy_score(y_test, y_pred))
# print("AUC:", roc_auc_score(y_test, cat.predict_proba(x_test)[:,1]))
# print(classification_report(y_test, y_pred))

In [ ]:
test_df = pd.read_csv('test.csv')
id = test_df['id']
test_df.drop(columns=['id'], inplace=True)

In [ ]:
print(x_train.columns.difference(test_df.columns))
print(test_df.columns.difference(x_train.columns))


In [ ]:
from sklearn.preprocessing import LabelEncoder

# Apply the same preprocessing to test_df as was done to x_train
for col in categorical_features:
    le = LabelEncoder()
    le.fit(x_train[col].astype(str))
    # Handle unknown values by mapping them to a default value
    test_df[col] = test_df[col].astype(str).apply(lambda x: le.transform([x])[0] if x in le.classes_ else 0)

# Scale numeric features
test_df[numeric_features] = scaler.transform(test_df[numeric_features])

test_proba = cat.predict_proba(test_df)[:, 1]
submission_df = pd.DataFrame({
    'id': id,
    'diagnosed_diabetes': test_proba
})

In [ ]:
submission_df.to_csv("submission.csv", index=False)
    